# ARM-CBA: Tuning & Feature Selection Experiments

## Objective
Optimize the CBA classifier through targeted feature validation, undersampling for class imbalance, and threshold tuning. CBA faces a unique challenge compared to LR and RF: its rule mining step (Apriori) depends on support thresholds, meaning rare classes can be mathematically invisible if they don't appear frequently enough in the training data.

## Experiment Plan
1. **Feature Validation:** Remove Accident Type and observe the impact on rule generation and classification
2. **Undersampling Inside CV Folds:** Balance class distribution before rule mining so minority causes (Signal, Environmental) can meet support thresholds
3. **Threshold Tuning on Undersampled Data:** Find the best support/confidence/lift configuration when the data is balanced

## Why CBA Tuning Differs From LR and RF
LR and RF tuning focused on hyperparameters (C, max_depth, n_estimators) that control model complexity. CBA has no equivalent knobs. Instead, CBA's performance is controlled by:
- **ARM thresholds** (support, confidence, lift) which determine how many rules are discovered
- **Class distribution** in the training data, which directly affects whether minority class rules can meet the support threshold
- **Feature set**, which determines what patterns ARM can find

The baseline notebook already showed that lowering thresholds from high (0.05/0.70/2.0) to low (0.005/0.40/1.5) improved F1 from 0.374 to 0.500 by generating more rules (6 to ~296). But all of those rules were dominated by Accident Type patterns. This notebook tests what happens when we remove that crutch and force the model to learn from environmental and operational features.

## Feature Selection Carry-Forward
Track Type vs Signalization and RUCC encoding decisions were established in LR tuning experiments and validated by Phase 2 statistical analysis. These are feature-level findings that apply regardless of model type:
- **Track Type** over Signalization (0.409 vs 0.378 F1 in LR)
- **RUCC_Metro_Adjacency** over RUCC_2023 (identical performance, better interpretability)

## Reference
Liu, B., Hsu, W., & Ma, Y. (1998). Integrating Classification and Association Rule Mining. *Proceedings of the Fourth International Conference on Knowledge Discovery and Data Mining (KDD-98)*, 80-86.

---

## Setup

In [ ]:
import pandas as pd
import numpy as np
import json
import joblib
import warnings

from mlxtend.frequent_patterns import apriori, association_rules
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    classification_report, f1_score, confusion_matrix,
    precision_score, recall_score
)

import matplotlib.pyplot as plt
import matplotlib.style as style
import seaborn as sns

warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=DeprecationWarning)
style.use('tableau-colorblind10')

plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 10

RANDOM_SEED = 521
np.random.seed(RANDOM_SEED)

CAUSE_ORDER = ['E', 'H', 'M', 'S', 'T']
CAUSE_LABELS = {
    'E': 'Environmental', 'H': 'Human Factor',
    'M': 'Mechanical', 'S': 'Signal', 'T': 'Track'
}

## CBA Classifier

Same implementation from the baseline notebook. Wraps ARM rule mining and first-match classification in a sklearn-compatible class with `fit()` and `predict()` methods. See baseline notebook for detailed step-by-step comments.

One addition: an optional `undersample` parameter. When enabled, the fit step balances the training data before mining rules by downsampling majority classes to match the smallest class count.

In [ ]:
class CBAClassifier(BaseEstimator, ClassifierMixin):
    """
    Classification Based on Associations (Liu et al., 1998).
    
    Mines association rules from training data using Apriori, ranks them
    by confidence/support/length, and classifies new instances by applying
    the first matching rule. Falls back to majority class when no rule matches.
    
    Parameters
    ----------
    min_support : float
        Minimum support threshold for Apriori.
    min_confidence : float
        Minimum confidence for filtering rules.
    min_lift : float
        Minimum lift for filtering rules.
    max_rule_length : int
        Maximum items in a frequent itemset.
    undersample : bool
        If True, balance training data before mining rules by downsampling
        majority classes to match the smallest class count.
    """
    
    def __init__(self, min_support=0.02, min_confidence=0.70, 
                 min_lift=1.5, max_rule_length=4, undersample=False):
        self.min_support = min_support
        self.min_confidence = min_confidence
        self.min_lift = min_lift
        self.max_rule_length = max_rule_length
        self.undersample = undersample
    
    def fit(self, X, y):
        """
        Mine association rules from training data and build ranked rule list.
        If undersample=True, balance classes before mining.
        """
        X_fit = X.copy()
        y_fit = y.copy()
        
        # Optional undersampling: balance classes before rule mining
        # This gives minority classes (Signal, Environmental) a fair chance
        # of meeting support thresholds by making them proportionally larger
        if self.undersample:
            min_count = y_fit.value_counts().min()
            balanced_idx = []
            for cls in y_fit.unique():
                cls_idx = y_fit[y_fit == cls].index.tolist()
                sampled = np.random.choice(cls_idx, size=min_count, replace=False)
                balanced_idx.extend(sampled)
            
            np.random.shuffle(balanced_idx)
            X_fit = X_fit.loc[balanced_idx].reset_index(drop=True)
            y_fit = y_fit.loc[balanced_idx].reset_index(drop=True)
        
        # One-hot encode features into binary transaction format
        X_ohe = pd.get_dummies(X_fit, columns=X_fit.columns.tolist(), dtype=bool)
        
        # One-hot encode target and append
        y_ohe = pd.get_dummies(y_fit, prefix='Cause', dtype=bool)
        self.cause_columns_ = y_ohe.columns.tolist()
        df_arm = pd.concat([X_ohe, y_ohe], axis=1)
        
        # Run Apriori
        frequent_itemsets = apriori(
            df_arm, 
            min_support=self.min_support, 
            use_colnames=True,
            max_len=self.max_rule_length
        )
        
        if len(frequent_itemsets) == 0:
            self.rules_ = pd.DataFrame()
            self.default_class_ = y_fit.mode()[0]
            self.n_rules_ = 0
            return self
        
        # Generate and filter rules
        rules = association_rules(frequent_itemsets, metric='lift', 
                                  min_threshold=self.min_lift)
        rules = rules[rules['confidence'] >= self.min_confidence]
        
        # Filter for Class Association Rules
        car_mask = rules['consequents'].apply(
            lambda x: len(x) == 1 and any(item in self.cause_columns_ for item in x)
        )
        cars = rules[car_mask].copy()
        
        # Rank: confidence desc, support desc, length asc
        cars['antecedent_length'] = cars['antecedents'].apply(len)
        cars = cars.sort_values(
            by=['confidence', 'support', 'antecedent_length'],
            ascending=[False, False, True]
        ).reset_index(drop=True)
        
        cars['predicted_cause'] = cars['consequents'].apply(
            lambda x: list(x)[0].replace('Cause_', '')
        )
        
        self.rules_ = cars
        self.n_rules_ = len(cars)
        self.feature_columns_ = X_ohe.columns.tolist()
        
        # Default rule: majority class from training data (Liu et al., 1998)
        # Uses original y (not undersampled) to reflect true class distribution
        self.default_class_ = y.mode()[0]
        
        return self
    
    def predict(self, X):
        """
        Classify each row by first matching rule. Default to majority class.
        """
        X_ohe = pd.get_dummies(X, columns=X.columns.tolist(), dtype=bool)
        X_ohe = X_ohe.reindex(columns=self.feature_columns_, fill_value=False)
        
        predictions = []
        
        if self.n_rules_ == 0:
            return np.array([self.default_class_] * len(X))
        
        # Pre-extract for speed
        rule_antecedents = []
        rule_predictions = []
        for _, rule in self.rules_.iterrows():
            antes = [a for a in rule['antecedents'] if a in self.feature_columns_]
            rule_antecedents.append(antes)
            rule_predictions.append(rule['predicted_cause'])
        
        for idx in range(len(X_ohe)):
            row = X_ohe.iloc[idx]
            matched = False
            
            for rule_idx in range(self.n_rules_):
                antes = rule_antecedents[rule_idx]
                if not antes:
                    continue
                if all(row[a] for a in antes):
                    predictions.append(rule_predictions[rule_idx])
                    matched = True
                    break
            
            if not matched:
                predictions.append(self.default_class_)
        
        return np.array(predictions)

print("CBAClassifier defined (with undersample option).")

## Helper Functions

In [ ]:
def run_cba_experiment(cba, X, y, n_splits=5):
    """
    Run stratified k-fold CV for CBA and return scores and metrics.
    """
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_SEED)
    
    scores = []
    rule_counts = []
    last_y_val = None
    last_y_pred = None
    
    for fold, (train_idx, val_idx) in enumerate(skf.split(X, y), 1):
        X_tr = X.iloc[train_idx]
        X_val = X.iloc[val_idx]
        y_tr = y.iloc[train_idx]
        y_val = y.iloc[val_idx]
        
        cba.fit(X_tr, y_tr)
        y_pred = cba.predict(X_val)
        f1 = f1_score(y_val, y_pred, average='weighted')
        scores.append(f1)
        rule_counts.append(cba.n_rules_)
        
        last_y_val = y_val
        last_y_pred = y_pred
    
    precision_per_class = precision_score(last_y_val, last_y_pred, average=None,
                                          labels=CAUSE_ORDER, zero_division=0)
    recall_per_class = recall_score(last_y_val, last_y_pred, average=None,
                                    labels=CAUSE_ORDER, zero_division=0)
    f1_per_class = f1_score(last_y_val, last_y_pred, average=None,
                            labels=CAUSE_ORDER, zero_division=0)
    
    per_class = pd.DataFrame({
        'Cause': CAUSE_ORDER,
        'Precision': precision_per_class.round(3),
        'Recall': recall_per_class.round(3),
        'F1': f1_per_class.round(3)
    })
    
    return {
        'scores': [float(s) for s in scores],
        'mean': float(np.mean(scores)),
        'std': float(np.std(scores)),
        'rule_counts': rule_counts,
        'per_class_metrics': per_class,
        'last_y_val': last_y_val,
        'last_y_pred': last_y_pred
    }


def save_experiment(results, features, experiment_name, filepath, **extra_fields):
    """
    Save experiment results to JSON.
    """
    output = {
        'experiment': experiment_name,
        'model': 'CBA (Classification Based on Associations)',
        'features': {
            'categorical': features['categorical'],
            'numeric': features['numeric'],
            'total': len(features['categorical']) + len(features['numeric'])
        },
        'cv_folds': 5,
        'cv_scores': results['scores'],
        'cv_mean': results['mean'],
        'cv_std': results['std'],
        'rule_counts': results['rule_counts'],
        'per_class_metrics': results['per_class_metrics'].to_dict('records'),
        'random_seed': RANDOM_SEED
    }
    output.update(extra_fields)
    
    with open(filepath, 'w') as f:
        json.dump(output, f, indent=4)
    
    print(f"Saved: {filepath}")


def compare_experiments(names, filepaths):
    """
    Load multiple experiment JSONs and display side-by-side comparison.
    """
    print("=" * 70)
    print(f"{'Experiment':<40} {'CV Mean':>10} {'CV Std':>10} {'Features':>8}")
    print("-" * 70)
    
    for name, fp in zip(names, filepaths):
        with open(fp, 'r') as f:
            data = json.load(f)
        
        mean = data.get('cv_mean', data.get('best_cv_mean', 0))
        std = data.get('cv_std', data.get('best_cv_std', 0))
        
        if isinstance(data.get('features'), dict):
            n_feat = data['features']['total']
        elif isinstance(data.get('features'), list):
            n_feat = len(data['features'])
        else:
            n_feat = '?'
        
        print(f"{name:<40} {mean:>10.4f} {std:>10.4f} {n_feat:>8}")
    
    print("=" * 70)


def plot_cv_comparison(experiments, title='CV Comparison'):
    """
    Plot per-fold F1 scores for multiple experiments.
    """
    fig, ax = plt.subplots(figsize=(8, 5))
    folds = np.arange(1, 6)
    
    markers = ['o', 's', '^', 'D', 'v']
    linestyles = ['-', '--', '-.', ':', '-']
    
    for i, (label, scores) in enumerate(experiments):
        mean = np.mean(scores)
        ax.plot(folds, scores, marker=markers[i % len(markers)], 
                linewidth=2, linestyle=linestyles[i % len(linestyles)],
                label=f'{label} (mean={mean:.4f})')
        ax.axhline(y=mean, linestyle=':', alpha=0.3)
    
    ax.set_xlabel('Fold')
    ax.set_ylabel('Weighted F1-Score')
    ax.set_title(title)
    ax.set_xticks(folds)
    ax.legend()
    plt.tight_layout()
    plt.show()


def plot_per_class_comparison(results_list, labels, title='Per-Class F1 Comparison'):
    """
    Side-by-side per-class F1 bars for multiple experiments.
    """
    fig, ax = plt.subplots(figsize=(8, 5))
    
    x = np.arange(len(CAUSE_ORDER))
    n = len(results_list)
    width = 0.8 / n
    hatches = ['//', '..', 'xx', '\\\\', '++']
    
    for i, (df, label) in enumerate(zip(results_list, labels)):
        offset = (i - n/2 + 0.5) * width
        bars = ax.bar(x + offset, df['F1'], width, label=label)
        for bar in bars:
            bar.set_hatch(hatches[i % len(hatches)])
    
    ax.set_xlabel('Cause Category')
    ax.set_ylabel('F1-Score')
    ax.set_title(title)
    ax.set_xticks(x)
    ax.set_xticklabels([f"{c}" for c in CAUSE_ORDER])
    ax.legend()
    ax.set_ylim(0, 1.0)
    plt.tight_layout()
    plt.show()

print("Helper functions defined.")

## Load Training Data

In [ ]:
df_train = pd.read_csv('../data/splits/train_80.csv', low_memory=False)

target = 'Cause_Category'
y_train = df_train[target].copy()

print(f"Training set: {len(df_train)} rows")
print(f"\nCause_Category distribution:")
print(y_train.value_counts())
print(f"\nSmallest class: {y_train.value_counts().min()} rows "
      f"({y_train.value_counts().min() / len(y_train):.1%})")

In [ ]:
# Load baseline results for comparison
with open('../results/cba/cba_baseline_results.json', 'r') as f:
    baseline_results = json.load(f)

print(f"Baseline CBA (Low Threshold) CV: {baseline_results.get('best_cv_mean', baseline_results.get('cv_mean', 'N/A'))}")

---

# Section 1: Feature Validation

## Experiment 1: Remove Accident Type

**What we are doing:** Removing Accident Type from the CBA feature set. This is the same experiment we ran on LR (drop of 0.127) and RF (drop of 0.130), now applied to CBA.

**Why we are doing it:** CBA has the strongest empirical case for removing Accident Type. Our Phase 2 ARM experiments proved that Accident Type dominates the entire rule space:
- With Accident Type: All rules are variations of Hwy-rail crossing to Mechanical and Side collision to Human Factor
- Without Accident Type: ARM produced zero rules at any threshold (support down to 0.005, confidence 0.50, lift 1.0)

That zero-rule result is the core problem this notebook needs to solve. Simply removing Accident Type without other changes will likely produce a model that defaults to majority class (H) for every prediction, because no rules can be mined from the remaining features at their natural class distribution.

**What to look for in the results:**
- **How many rules are generated:** If zero (as we saw in Phase 2), the model is just a majority class predictor
- **F1 score:** Expected to drop significantly, possibly below LR and RF since CBA depends more heavily on strong feature-to-target associations
- **Rule coverage:** What percentage of rows match any rule vs defaulting to H
- **This result motivates Experiment 2:** If Experiment 1 confirms the zero-rule problem, undersampling becomes the necessary intervention

In [ ]:
# Experiment 1: 5 features, no Accident Type, same low thresholds as baseline
cat_features_exp1 = ['Weather Condition', 'Track Type', 'Visibility', 'Region', 
                      'RUCC_Metro_Adjacency']
num_features_exp1 = []

X_exp1 = df_train[cat_features_exp1].copy()

# Use the baseline's best low thresholds
cba_exp1 = CBAClassifier(
    min_support=0.005, 
    min_confidence=0.40, 
    min_lift=1.5, 
    max_rule_length=4,
    undersample=False
)

print("Experiment 1: CBA without Accident Type (no undersampling)")
print("-" * 55)
results_exp1 = run_cba_experiment(cba_exp1, X_exp1, y_train)

for fold, (score, n_rules) in enumerate(zip(results_exp1['scores'], results_exp1['rule_counts']), 1):
    print(f"  Fold {fold}: Weighted F1 = {score:.4f}  (Rules: {n_rules})")
print(f"\nCV: {results_exp1['mean']:.4f} +/- {results_exp1['std']:.4f}")

In [ ]:
# Examine what rules (if any) were found
if cba_exp1.n_rules_ > 0:
    print(f"Rules mined (last fold): {cba_exp1.n_rules_}")
    print(f"Default class: {cba_exp1.default_class_}")
    print(f"\nRule count by predicted cause:")
    print(cba_exp1.rules_['predicted_cause'].value_counts())
    print(f"\nTop 10 rules:")
    top_rules = cba_exp1.rules_[['antecedents', 'predicted_cause', 'confidence', 
                                   'support', 'lift']].head(10).copy()
    top_rules['antecedents'] = top_rules['antecedents'].apply(lambda x: ', '.join(sorted(x)))
    print(top_rules.to_string(index=False))
else:
    print(f"Rules mined: 0")
    print(f"Default class: {cba_exp1.default_class_}")
    print(f"\nNo rules generated. Model defaults to majority class (H) for all predictions.")
    print(f"This confirms Phase 2 ARM findings: without Accident Type,")
    print(f"the remaining features cannot meet support thresholds at natural class distribution.")

In [ ]:
save_experiment(
    results_exp1,
    features={'categorical': cat_features_exp1, 'numeric': num_features_exp1},
    experiment_name='CBA without Accident Type (no undersampling)',
    filepath='../results/cba/cba_exp_no_accident_type.json',
    thresholds={'min_support': 0.005, 'min_confidence': 0.40, 'min_lift': 1.5},
    undersample=False,
    notes='Expected to confirm Phase 2 zero-rule finding. Motivates undersampling experiment.'
)

compare_experiments(
    names=['Baseline (6 feat, w/ AT)', 'Without AT (no undersampling)'],
    filepaths=['../results/cba/cba_baseline_results.json', '../results/cba/cba_exp_no_accident_type.json']
)

### Experiment 1 Findings

*[Fill in after running]*

**Cross-model comparison:**
- LR with AT: 0.536 -> without: 0.409 (drop: 0.127)
- RF with AT: 0.551 -> without: 0.421 (drop: 0.130)
- CBA with AT: 0.500 -> without: *[fill in]* (drop: *[fill in]*)

**Rules generated:** *[How many? For which causes?]*

**Decision:** *[Does this motivate undersampling?]*

---

## Experiment 2: Undersampling Inside CV Folds

**What we are doing:** Before mining rules in each CV fold, we downsample majority classes (H, M, T) to match the smallest class count (Signal, ~700 rows in training). This creates a balanced dataset where each cause category represents exactly 20% of the data, then we mine rules from that balanced data.

**Why we are doing it:** ARM's Apriori algorithm uses a minimum support threshold to decide which patterns are frequent enough to become rules. Support is defined as the fraction of rows containing a pattern. When Signal represents only 2.5% of the data, a rule predicting Signal can have at most 2.5% support, which falls below most reasonable thresholds. By balancing the classes, Signal patterns now have the same opportunity to meet support thresholds as Human Factor patterns.

**How undersampling works in practice:**
1. Count the smallest class in the training fold (Signal, roughly 550-700 rows depending on the fold)
2. Randomly sample that many rows from each of the other 4 classes
3. Combine into a balanced dataset (roughly 2,750-3,500 total rows)
4. Mine rules from this balanced data
5. Predict on the ORIGINAL (unbalanced) validation fold

Step 5 is critical: we balance the data for rule mining only, not for evaluation. The validation fold retains its natural class distribution so our F1 scores reflect real-world performance.

**What to look for in the results:**
- **Rule diversity:** Do we now get rules for all 5 cause categories, not just M and H?
- **Signal and Environmental rules:** These were impossible without undersampling. Do they appear?
- **F1 score:** May go up or down. More rules means better coverage, but rules mined from undersampled data may be less reliable on the natural distribution.
- **Coverage:** What percentage of rows now match a rule vs defaulting?

In [ ]:
# Experiment 2: Same features, same thresholds, but with undersampling
cba_exp2 = CBAClassifier(
    min_support=0.005, 
    min_confidence=0.40, 
    min_lift=1.5, 
    max_rule_length=4,
    undersample=True
)

print("Experiment 2: CBA without Accident Type (with undersampling)")
print("-" * 55)
results_exp2 = run_cba_experiment(cba_exp2, X_exp1, y_train)

for fold, (score, n_rules) in enumerate(zip(results_exp2['scores'], results_exp2['rule_counts']), 1):
    print(f"  Fold {fold}: Weighted F1 = {score:.4f}  (Rules: {n_rules})")
print(f"\nCV: {results_exp2['mean']:.4f} +/- {results_exp2['std']:.4f}")

In [ ]:
# Examine rules from undersampled model
if cba_exp2.n_rules_ > 0:
    print(f"Rules mined (last fold): {cba_exp2.n_rules_}")
    print(f"Default class: {cba_exp2.default_class_}")
    print(f"\nRule count by predicted cause:")
    print(cba_exp2.rules_['predicted_cause'].value_counts())
    print(f"\nTop 15 rules:")
    top_rules = cba_exp2.rules_[['antecedents', 'predicted_cause', 'confidence', 
                                   'support', 'lift']].head(15).copy()
    top_rules['antecedents'] = top_rules['antecedents'].apply(lambda x: ', '.join(sorted(x)))
    print(top_rules.to_string(index=False))
else:
    print(f"Rules mined: 0")
    print(f"Even with undersampling, no rules generated at these thresholds.")

In [ ]:
# Per-class metrics
print("Per-class metrics (undersampled CBA, no AT):")
print(results_exp2['per_class_metrics'].to_string(index=False))

print(f"\nClassification report:")
print(classification_report(results_exp2['last_y_val'], results_exp2['last_y_pred'], 
                             labels=CAUSE_ORDER, zero_division=0))

save_experiment(
    results_exp2,
    features={'categorical': cat_features_exp1, 'numeric': num_features_exp1},
    experiment_name='CBA without AT (undersampled)',
    filepath='../results/cba/cba_exp_undersampled.json',
    thresholds={'min_support': 0.005, 'min_confidence': 0.40, 'min_lift': 1.5},
    undersample=True
)

compare_experiments(
    names=['Baseline (w/ AT)', 'No AT (no undersample)', 'No AT (undersampled)'],
    filepaths=[
        '../results/cba/cba_baseline_results.json',
        '../results/cba/cba_exp_no_accident_type.json',
        '../results/cba/cba_exp_undersampled.json'
    ]
)

### Experiment 2 Findings

*[Fill in after running]*

**Rule diversity:** *[Did we get rules for all 5 causes?]*

**Coverage improvement:** *[What % of rows now match rules?]*

**F1 impact:** *[Did undersampling help or hurt overall?]*

**Decision:** *[Does undersampling produce a viable CBA model?]*

---

## Experiment 3: Threshold Tuning on Undersampled Data

**What we are doing:** Testing multiple support/confidence combinations on the undersampled data to find the best threshold configuration. This is CBA's equivalent of GridSearchCV - instead of searching over model hyperparameters, we search over rule mining thresholds.

**Why we are doing it:** The thresholds we used in Experiment 2 (0.005/0.40/1.5) were optimized for the original data with Accident Type. Undersampled data has different statistical properties: each class represents 20% of the data instead of 2.5-39%, which fundamentally changes what support levels are achievable. Thresholds that were appropriate for imbalanced data may be too low or too high for balanced data.

**What to look for in the results:**
- **Higher support may now work:** With balanced classes, even minority class patterns can hit 5-10% support. Higher support means more reliable rules.
- **Confidence vs coverage tradeoff:** Higher confidence produces fewer but more accurate rules. Lower confidence gives more rules but more wrong predictions. The optimal point balances coverage against accuracy.
- **Rule count sweet spot:** Too few rules means most rows default to majority class. Too many rules means unreliable rules fire before reliable ones. There's a middle ground.

In [ ]:
# Threshold grid search for undersampled CBA
threshold_configs = [
    {'min_support': 0.01,  'min_confidence': 0.30, 'min_lift': 1.5, 'label': 'S=0.01, C=0.30'},
    {'min_support': 0.01,  'min_confidence': 0.40, 'min_lift': 1.5, 'label': 'S=0.01, C=0.40'},
    {'min_support': 0.01,  'min_confidence': 0.50, 'min_lift': 1.5, 'label': 'S=0.01, C=0.50'},
    {'min_support': 0.005, 'min_confidence': 0.30, 'min_lift': 1.5, 'label': 'S=0.005, C=0.30'},
    {'min_support': 0.005, 'min_confidence': 0.40, 'min_lift': 1.5, 'label': 'S=0.005, C=0.40'},
    {'min_support': 0.005, 'min_confidence': 0.50, 'min_lift': 1.5, 'label': 'S=0.005, C=0.50'},
    {'min_support': 0.05,  'min_confidence': 0.30, 'min_lift': 1.5, 'label': 'S=0.05, C=0.30'},
    {'min_support': 0.05,  'min_confidence': 0.40, 'min_lift': 1.5, 'label': 'S=0.05, C=0.40'},
    {'min_support': 0.05,  'min_confidence': 0.50, 'min_lift': 1.5, 'label': 'S=0.05, C=0.50'},
    {'min_support': 0.10,  'min_confidence': 0.30, 'min_lift': 1.5, 'label': 'S=0.10, C=0.30'},
    {'min_support': 0.10,  'min_confidence': 0.40, 'min_lift': 1.5, 'label': 'S=0.10, C=0.40'},
    {'min_support': 0.10,  'min_confidence': 0.50, 'min_lift': 1.5, 'label': 'S=0.10, C=0.50'},
]

print(f"Testing {len(threshold_configs)} threshold configurations (undersampled)...")
print(f"{'Config':<25} {'CV Mean':>10} {'CV Std':>10} {'Avg Rules':>10}")
print("-" * 58)

threshold_results = []

for config in threshold_configs:
    cba_test = CBAClassifier(
        min_support=config['min_support'],
        min_confidence=config['min_confidence'],
        min_lift=config['min_lift'],
        max_rule_length=4,
        undersample=True
    )
    
    results = run_cba_experiment(cba_test, X_exp1, y_train)
    avg_rules = np.mean(results['rule_counts'])
    
    print(f"{config['label']:<25} {results['mean']:>10.4f} {results['std']:>10.4f} {avg_rules:>10.1f}")
    
    threshold_results.append({
        'config': config,
        'results': results,
        'avg_rules': avg_rules
    })

# Find best
best_idx = np.argmax([r['results']['mean'] for r in threshold_results])
best_config = threshold_results[best_idx]
print(f"\nBest: {best_config['config']['label']} "
      f"(F1 = {best_config['results']['mean']:.4f}, "
      f"Avg rules = {best_config['avg_rules']:.1f})")

In [ ]:
# Detailed analysis of best threshold config
best_cfg = best_config['config']
cba_best = CBAClassifier(
    min_support=best_cfg['min_support'],
    min_confidence=best_cfg['min_confidence'],
    min_lift=best_cfg['min_lift'],
    max_rule_length=4,
    undersample=True
)

results_best = best_config['results']

print(f"Best Config: {best_cfg['label']}")
print(f"CV: {results_best['mean']:.4f} +/- {results_best['std']:.4f}")
print(f"\nPer-class metrics:")
print(results_best['per_class_metrics'].to_string(index=False))

print(f"\nClassification report:")
print(classification_report(results_best['last_y_val'], results_best['last_y_pred'], 
                             labels=CAUSE_ORDER, zero_division=0))

In [ ]:
# Refit best config to inspect rules
cba_best.fit(X_exp1, y_train)

if cba_best.n_rules_ > 0:
    print(f"Rules mined (full training set): {cba_best.n_rules_}")
    print(f"Default class: {cba_best.default_class_}")
    print(f"\nRule count by predicted cause:")
    print(cba_best.rules_['predicted_cause'].value_counts())
    print(f"\nTop 15 rules:")
    top_rules = cba_best.rules_[['antecedents', 'predicted_cause', 'confidence', 
                                   'support', 'lift']].head(15).copy()
    top_rules['antecedents'] = top_rules['antecedents'].apply(lambda x: ', '.join(sorted(x)))
    print(top_rules.to_string(index=False))
else:
    print("No rules generated with best config.")

In [ ]:
save_experiment(
    results_best,
    features={'categorical': cat_features_exp1, 'numeric': num_features_exp1},
    experiment_name=f'CBA Tuned (undersampled, {best_cfg["label"]})',
    filepath='../results/cba/cba_tuned_undersampled.json',
    thresholds={
        'min_support': best_cfg['min_support'],
        'min_confidence': best_cfg['min_confidence'],
        'min_lift': best_cfg['min_lift']
    },
    undersample=True,
    n_configs_tested=len(threshold_configs)
)

### Experiment 3 Findings

*[Fill in after running]*

**Best thresholds:** *[Support, confidence, lift]*

**Did tuning help over Experiment 2 defaults?** *[Compare F1]*

**Rule characteristics:** *[How many rules? Coverage across cause categories?]*

---

## Visualizations

In [ ]:
# Compare all CBA experiments
plot_cv_comparison(
    experiments=[
        ('No AT (no undersample)', results_exp1['scores']),
        ('No AT (undersampled)', results_exp2['scores']),
        (f'Tuned ({best_cfg["label"]})', results_best['scores'])
    ],
    title='CBA Experiments: Impact of Undersampling and Threshold Tuning'
)

In [ ]:
# Per-class F1 comparison
plot_per_class_comparison(
    [results_exp1['per_class_metrics'], results_exp2['per_class_metrics'], 
     results_best['per_class_metrics']],
    ['No undersample', 'Undersampled', f'Tuned ({best_cfg["label"]})'],
    title='CBA: Per-Class F1 Across Experiments'
)

In [ ]:
# Confusion matrix for best model
cm = confusion_matrix(results_best['last_y_val'], results_best['last_y_pred'], 
                       labels=CAUSE_ORDER)

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='cividis',
            xticklabels=CAUSE_ORDER, yticklabels=CAUSE_ORDER, ax=ax)
ax.set_title(f'CBA Tuned - Confusion Matrix')
ax.set_ylabel('True Label')
ax.set_xlabel('Predicted Label')

plt.tight_layout()
plt.show()

---

## Save Best Model

In [ ]:
# Save best CBA model and rules
joblib.dump(cba_best, '../models/cba_tuned_model.pkl')
print(f"Saved: ../models/cba_tuned_model.pkl")

# Save rules to CSV
if cba_best.n_rules_ > 0:
    rules_export = cba_best.rules_[['antecedents', 'predicted_cause', 
                                     'confidence', 'support', 'lift']].copy()
    rules_export['antecedents'] = rules_export['antecedents'].apply(
        lambda x: ', '.join(sorted(x))
    )
    rules_export.to_csv('../data/processed/arm_rules/cba_tuned_rules.csv', index=False)
    print(f"Saved {len(rules_export)} rules to cba_tuned_rules.csv")

---

## Final Summary

In [ ]:
# Load and compare all CBA experiments
import os

cba_results_dir = '../results/cba/'
all_files = sorted([f for f in os.listdir(cba_results_dir) if f.endswith('.json')])

print("=" * 75)
print(f"{'Experiment':<45} {'CV Mean':>10} {'CV Std':>10} {'Features':>8}")
print("-" * 75)

for f in all_files:
    filepath = os.path.join(cba_results_dir, f)
    with open(filepath, 'r') as fh:
        data = json.load(fh)
    
    name = data.get('experiment', f.replace('.json', ''))
    mean = data.get('cv_mean', data.get('best_cv_mean', 0))
    std = data.get('cv_std', data.get('best_cv_std', 0))
    
    if isinstance(data.get('features'), dict):
        n_feat = data['features']['total']
    elif isinstance(data.get('features'), list):
        n_feat = len(data['features'])
    else:
        n_feat = '?'
    
    print(f"{name:<45} {mean:>10.4f} {std:>10.4f} {n_feat:>8}")

print("=" * 75)

## Conclusions

*[Fill in after all experiments]*

**Best CBA configuration:** *[Thresholds, undersampling, features]*

**Key findings:**
1. *[Impact of removing Accident Type on CBA specifically]*
2. *[Did undersampling solve the zero-rule problem?]*
3. *[Threshold tuning impact on undersampled data]*
4. *[Rule diversity: did we finally get rules for all 5 causes?]*
5. *[How does tuned CBA compare to tuned LR and RF?]*

**Final tuned model saved to:** `../models/cba_tuned_model.pkl`
**All experiment results saved to:** `../results/cba/`